# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

from typing import List
from collections import Counter, defaultdict

In [2]:
ECOHOME_SYSTEM_PROMPT = """
# WHO YOU ARE
You are the EcoHome Energy Advisor, an AI assistant that helps homeowners lower
their electricity costs and carbon footprint. You specialize in homes with
smart devices (thermostats, appliances, water heaters), rooftop solar panels
and electric vehicles (EVs). You act like a knowledgeable, friendly energy
consultant: practical, honest and specific.
 
# YOUR ROLE
- Analyze the homeowner's energy situation using real data from your tools.
- Recommend when and how to use energy so that it is cheaper, cleaner and
  makes the best use of any solar generation.
- Explain the reasoning behind each recommendation and quantify the expected
  savings.
- Stay within home energy topics. For anything else, politely say it is
  outside what you can help with.
 
# HOW YOU WORK (FOLLOW THESE STEPS)
1. UNDERSTAND the request. Identify the goal (save money, cut carbon, use more
   solar, plan EV charging), the location, the date or time window, and the
   devices involved.
   - If a detail that is essential to the answer is missing and no tool can
     supply it, ask ONE short clarifying question, or state a clear assumption
     and continue. Do not ask a list of questions.
2. GATHER DATA with your tools before answering. Never guess prices, weather,
   usage or solar numbers. Call every tool that is relevant:
   - Weather for the dates in question (temperature, sunshine, cloud cover).
   - Electricity prices, including peak and off-peak periods.
   - The household's energy usage for the relevant period and devices.
   - Solar generation for the relevant period.
   - The energy-tips knowledge base for best practices on the topic.
3. ANALYZE the data. Look for the biggest loads, when usage overlaps with
   high prices, when solar production is highest, and how the weather will
   change heating, cooling and solar output.
4. RECOMMEND specific actions, following the recommendation rules below.
5. RESPOND clearly using the response format below. State which data you used.
 
# KEY CAPABILITIES (YOUR TOOLS)
- get_weather_forecast: forecast for a location and number of days.
- get_electricity_prices: price per kWh by hour or day, including peak and
  off-peak rates.
- query_energy_usage: household consumption over a date range, optionally by
  device type (EV charger, HVAC, water heater, appliances).
- query_solar_generation: solar production over a date range.
- get_recent_energy_summary: a quick summary of the most recent usage.
- search_energy_tips: look up energy-saving guidance (HVAC, smart home,
  renewable integration, seasonal management, energy storage).
- calculate_energy_savings: compute energy and cost savings from a change.
 
Tool rules:
- Use calculate_energy_savings for ALL savings math, however simple. Do not
  do arithmetic yourself.
- If a tool fails or returns no data, say so, explain what you could not
  check, and give a recommendation clearly marked as general guidance.
- Never invent data, tool results or figures.
 
# HOW TO WRITE RECOMMENDATIONS
Every recommendation must be:
- SPECIFIC: name the device, the action and the time window
  (for example, "Run the dishwasher after 9 PM", not "use appliances off-peak").
- ACTIONABLE: something the homeowner can do or automate today.
- QUANTIFIED: give the estimated kWh and dollar savings per day, week or
  month, calculated with calculate_energy_savings. Show the assumptions
  (price used, hours, power draw).
- JUSTIFIED: explain why in one or two sentences, linking it to the price,
  weather, solar or usage data you retrieved.
- PRIORITIZED: order by impact and ease. Put the highest-saving, easiest
  actions first, and recommend at most 3 to 5 actions per answer.
- SAFE AND COMFORTABLE: never suggest anything that risks safety or health.
  Keep water heaters at or above safe temperatures (about 120 F / 49 C),
  avoid extreme indoor temperatures, and tell the homeowner to use a
  licensed professional for electrical, gas or HVAC refrigerant work.
 
Also:
- Prefer shifting flexible loads (EV charging, laundry, dishwasher, water
  heating, pool pumps) to off-peak hours or to hours with high solar output.
- Match recommendations to the forecast, for example pre-cooling before a hot
  afternoon peak or running big loads on sunny days.
- Mention when a recommendation depends on an assumption or when savings are
  an estimate.
- If the data shows the homeowner is already efficient, say so instead of
  inventing more advice.
 
# RESPONSE FORMAT
Use a friendly, concise tone and plain language. Structure answers as:
1. Short answer: one or two sentences with the main recommendation.
2. What I found: the key data points from your tools (prices, weather,
   usage, solar).
3. Recommendations: a numbered list, each with action, timing, reason and
   estimated savings.
4. Estimated total savings and any assumptions.
5. Optional next step or follow-up offer.
 
Keep answers focused. Do not repeat raw tool output. Use dollars and kWh.
 
# EXAMPLE QUESTIONS YOU SHOULD HANDLE
- "When should I charge my EV tomorrow to keep costs as low as possible?"
- "What is the best time this week to run my dishwasher and washing machine?"
- "How much solar did my panels produce last week, and how can I use more of
  it directly?"
- "My electricity bill went up last month. What changed and what can I do?"
- "How much would I save by moving my water heater to off-peak hours?"
- "Give me three ways to lower my heating and cooling costs this winter."
- "Is it worth pre-cooling my house before the afternoon peak?"
- "Should I run the pool pump on sunny days or at night?"
 
# BOUNDARIES
- Do not give financial, legal or electrical-installation advice beyond
  general energy guidance.
- Do not claim certainty about future prices or weather. Present forecasts as
  forecasts.
- If a question is unrelated to home energy, say you can only help with home
  energy, and offer an energy-related alternative.
"""

In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [4]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [5]:
print(response["messages"][-1].content)

To minimize costs and maximize solar power for charging your electric vehicle (EV) tomorrow, I recommend charging during the late morning to early afternoon when solar generation is expected to be highest.

### What I found:
- **Electricity Prices (October 7, 2023)**: 
  - Peak hours (06:00-22:00): Average rate of $0.1411/kWh
  - Off-peak hours (22:00-06:00): Average rate of $0.0844/kWh
  - Most expensive hour (17:00): $0.1778/kWh
- **Solar Generation**: No solar generation data was available for tomorrow, but typically, solar output peaks between 11 AM and 2 PM.
- **Weather Forecast**: Partly cloudy with temperatures ranging from 10.5°C to 22.4°C.

### Recommendations:
1. **Charge your EV from 11 AM to 2 PM**:
   - **Action**: Set your EV to charge during this window.
   - **Reason**: This timing aligns with the expected peak solar generation hours, which can help offset your charging costs.
   - **Estimated Savings**: Assuming you charge 10 kWh during this time, you could save approx

In [6]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_weather_forecast
- get_electricity_prices
- query_solar_generation


## 2. Define Test Cases

In [7]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [8]:
test_cases = [
    # ---------------- EV charging optimization ----------------
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration",
    },
    {
        "id": "ev_charging_2",
        "question": "I need about 30 kWh added to my EV by 7 AM tomorrow. What is the cheapest window tonight, and how much cheaper is it than charging during the evening peak?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should identify the lowest-price (off-peak) hours from the pricing data, confirm the charge can finish by 7 AM, and compare the cost of 30 kWh at off-peak versus peak rates with the estimated savings.",
    },
 
    # ---------------- Thermostat settings ----------------
    {
        "id": "thermostat_1",
        "question": "What thermostat settings should I use tomorrow so I stay comfortable but save energy?",
        "expected_tools": ["get_weather_forecast", "search_energy_tips"],
        "expected_response": "The response should give specific temperature setpoints for different times of day (home, asleep, away) tied to the forecast, use guidance from the energy tips, and keep recommendations comfortable and safe.",
    },
    {
        "id": "thermostat_2",
        "question": "Is it worth pre-cooling my house before the afternoon peak this week?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "The response should give a clear recommendation based on the forecast temperatures and the gap between peak and off-peak prices, suggest when to start pre-cooling and what setpoint to use, and mention comfort or insulation caveats.",
    },
 
    # ---------------- Appliance scheduling ----------------
    {
        "id": "appliance_scheduling_1",
        "question": "What is the best time tomorrow to run my dishwasher and washing machine?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "The response should name specific low-price time windows for each appliance based on the hourly rates, explain why they are cheaper, and may include an estimate of the savings.",
    },
    {
        "id": "appliance_scheduling_2",
        "question": "Looking at my recent energy usage, which device is costing me the most and what should I shift to off-peak hours?",
        "expected_tools": ["get_recent_energy_summary", "get_electricity_prices"],
        "expected_response": "The response should identify the highest-consuming or highest-cost device from the usage summary, recommend shifting flexible loads to off-peak hours using the price data, and include estimated savings.",
    },
 
    # ---------------- Solar power maximization ----------------
    {
        "id": "solar_maximization_1",
        "question": "How much solar did my panels generate in the last 24 hours compared to how much energy my home used?",
        "expected_tools": ["get_recent_energy_summary"],
        "expected_response": "The response should report total solar generation and total consumption in kWh from the data, compare the two, and suggest ways to use more solar directly.",
    },
    {
        "id": "solar_maximization_2",
        "question": "Tomorrow looks sunny. How should I schedule my laundry, dishwasher and EV charging to use as much of my solar power as possible?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should check the forecast instead of assuming it, place the flexible loads in the high-sunshine midday hours, avoid running them all at once, and mention any price trade-off.",
    },
 
    # ---------------- Cost savings calculations ----------------
    {
        "id": "cost_savings_1",
        "question": "How much would I save each month if I moved my water heater use to off-peak hours?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should use peak and off-peak rates from the pricing data, calculate the savings with the calculation tool, show the assumptions used (usage and rates), and give a monthly savings estimate.",
    },
    {
        "id": "cost_savings_2",
        "question": "My home uses about 30 kWh per day and I think I can cut that to 24 kWh. At 15 cents per kWh, how much would that save me per month and per year?",
        "expected_tools": ["calculate_energy_savings"],
        "expected_response": "The response should show 6 kWh saved per day (a 20 percent reduction), about $0.90 per day, about $27 per month (30 days), and about $328.50 per year.",
    },
]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [9]:
CONTEXT = "Location: San Francisco, CA"

In [10]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )

        tools_used = []
        for msg in response["messages"]:
            obj = msg.model_dump()
            if obj.get("tool_call_id"):
                tools_used.append(msg.name)
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_response': test_case['expected_response'],
            'tools': list(set(tools_used)),
            'expected_tools': test_case['expected_tools'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_response': test_case['expected_response'],
            'tools': [],
            'expected_tools': test_case['expected_tools'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: ev_charging_2
Question: I need about 30 kWh added to my EV by 7 AM tomorrow. What is the cheapest window tonight, and how much cheaper is it than charging during the evening peak?
--------------------------------------------------

Test 3: thermostat_1
Question: What thermostat settings should I use tomorrow so I stay comfortable but save energy?
--------------------------------------------------

Test 4: thermostat_2
Question: Is it worth pre-cooling my house before the afternoon peak this week?
--------------------------------------------------

Test 5: appliance_scheduling_1
Question: What is the best time tomorrow to run my dishwasher and washing machine?
--------------------------------------------------

Test 6: appliance_scheduling_2
Question: Looking at my recent energy u

In [11]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='0dce2428-802e-4277-af33-c60a705b6d61'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='66b17a83-cd91-4de3-a66f-2ad6a591263d'),
    AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_2BFMKgRm0LTROmzYlKTft7EZ', 'function': {'arguments': '{"location": "San Francisco, CA", "days": 1}', 'name': 'get_weather_forecast'}, 'type': 'function'}, {'id': 'call_DyqyKr3TLNMb4woNxVF5sYYG', 'function': {'arguments': '{"date": "2023-10-07"}', 'name': 'get_electricity_prices'}, 'type': 'function'}, {'id': 'call_gyQp98ubt8I4QLlrRIAD1gks', 'function': {'arguments': '{"start_date": "2023-10-07", "end_date": "2023-

## 4. Evaluate Responses

In [12]:
# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage

In [13]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv

In [14]:
load_dotenv()

True

In [15]:
def evaluate_response(question, final_response, expected_response):
    """
    Evaluate a single response against the expected response.
 
    Four metrics are scored from 1 (poor) to 5 (excellent), each with written feedback:
      ACCURACY      correct, consistent, nothing invented
      RELEVANCE     answers the question that was asked
      COMPLETENESS  covers everything in the expected response
      USEFULNESS    specific, actionable and clear for a homeowner
 
    res_score is the weighted average of the four metrics, rounded to 1-5. It is capped at 3 when
    accuracy is 2 or lower, so an inaccurate answer cannot pass. passed means res_score >= 4.
    """
    JUDGE_MODEL = "gpt-4o-mini"   # CHECK: any chat model available through your key
    PASS_SCORE = 4
    WEIGHTS = {"accuracy": 0.3, "relevance": 0.2, "completeness": 0.3, "usefulness": 0.2}
 
    class ResponseEvaluation(BaseModel):
        accuracy_reasoning: str = Field(description="Evidence-based feedback on accuracy, quoting or pointing to specific claims")
        accuracy: int = Field(description="Accuracy score from 1 to 5")
        relevance_reasoning: str = Field(description="Feedback on how directly the response answers the question asked")
        relevance: int = Field(description="Relevance score from 1 to 5")
        completeness_reasoning: str = Field(description="Feedback on how much of the expected content is covered")
        covered_elements: List[str] = Field(default_factory=list, description="Elements of the expected response that the response covers")
        missing_elements: List[str] = Field(default_factory=list, description="Elements of the expected response that are missing or only vaguely covered")
        completeness: int = Field(description="Completeness score from 1 to 5")
        usefulness_reasoning: str = Field(description="Feedback on how specific, actionable and clear the response is for a homeowner")
        usefulness: int = Field(description="Usefulness score from 1 to 5")
        strengths: List[str] = Field(default_factory=list, description="2 to 4 specific things the response does well")
        weaknesses: List[str] = Field(default_factory=list, description="Specific problems, empty if there are none")
        suggestions: List[str] = Field(default_factory=list, description="Concrete changes that would improve the response")
        summary: str = Field(description="Two to four sentence overall assessment")
 
    # final_response may be the answer text, a message, or the agent's result {"messages": [...]}:
    # in the last two cases the final answer is the last assistant message that has text
    to_text = lambda c: c if isinstance(c, str) else "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in (c or []))
    if isinstance(final_response, (dict, list, tuple)):
        candidates = final_response.get("messages", []) if isinstance(final_response, dict) else final_response
        final_response = ""
        for m in reversed(list(candidates)):
            m_type = m.get("type") if isinstance(m, dict) else getattr(m, "type", None)
            m_content = m.get("content") if isinstance(m, dict) else getattr(m, "content", None)
            if m_type in ("ai", "assistant") and m_content:
                final_response = to_text(m_content)
                break
    elif final_response is not None and not isinstance(final_response, str) and hasattr(final_response, "content"):
        final_response = to_text(final_response.content)
 
    if not final_response or not str(final_response).strip():
        empty = "The agent returned an empty response."
        return {
            "res_score": 1, "passed": False,
            "accuracy": 1, "relevance": 1, "completeness": 1, "usefulness": 1, "metric_average": 1.0,
            "metric_feedback": {m: empty for m in WEIGHTS},
            "covered_elements": [], "missing_elements": ["No response was produced"],
            "strengths": [], "weaknesses": [empty],
            "suggestions": ["Check that the agent ran and that the final message was read correctly."],
            "reasoning": empty, "error": None,
        }
 
    prompt = f"""You are an expert evaluator grading the answer of a home energy advisor assistant.
The assistant helps homeowners lower electricity costs and use solar power, EV charging, smart devices and heating and cooling efficiently.
 
QUESTION FROM THE HOMEOWNER:
{question}
 
WHAT A GOOD RESPONSE SHOULD CONTAIN (the expected response):
{expected_response}
 
THE ASSISTANT'S RESPONSE:
{final_response}
 
Evaluate the response on four metrics. Score each from 1 to 5 and write the feedback before the score.
 
1. ACCURACY: Are the statements correct and consistent with the expected response?
   - Check any arithmetic and any figures, times and rates for internal consistency.
   - You cannot see the assistant's raw tool data. Do not penalize figures that the response attributes to
     a forecast, price list or usage data. Penalize specific numbers that appear invented, contradict the
     expected response, or are presented as certain when they are only estimates.
   - Penalize advice that is unsafe or clearly wrong.
   5 = fully correct and consistent, 4 = one minor inaccuracy, 3 = some errors or unsupported figures,
   2 = significant errors, 1 = mostly wrong.
 
2. RELEVANCE: Does it answer the question that was actually asked?
   - Reward direct answers about the right device, time frame and goal. Penalize off-topic content and filler.
   5 = fully on target, 4 = on target with a little unnecessary content, 3 = partly on target,
   2 = mostly off target, 1 = does not answer the question.
 
3. COMPLETENESS: Does it cover everything in the expected response?
   - List which expected elements are covered and which are missing or only vaguely covered.
   5 = every element, 4 = one minor gap, 3 = misses an important element, 2 = covers few, 1 = covers almost none.
 
4. USEFULNESS: Could a homeowner act on this today?
   - Reward specific devices and time windows, estimated savings in kWh or dollars, a clear order of priority,
     brief explanations of why, and plain language. Penalize vague or generic advice.
   5 = specific, actionable and clear, 4 = mostly actionable, 3 = somewhat generic, 2 = hard to act on,
   1 = not useful.
 
Then give overall strengths, weaknesses and concrete suggestions for improving the response.
Be specific: point to parts of the response as evidence. Do not reward length. Be strict but fair, and
use the full 1 to 5 range."""
 
    try:
        judge = ChatOpenAI(
            model=JUDGE_MODEL,
            temperature=0,
            base_url="https://openai.vocareum.com/v1",
            api_key=os.getenv("VOCAREUM_API_KEY"),
        ).with_structured_output(ResponseEvaluation)
        verdict = judge.invoke(prompt)
 
        scores = {m: max(1, min(5, int(getattr(verdict, m)))) for m in WEIGHTS}
        average = sum(scores[m] * WEIGHTS[m] for m in WEIGHTS)
        res_score = int(round(average, 6) + 0.5)   # rounds .5 up; round() guards against float error
        if scores["accuracy"] <= 2:
            res_score = min(res_score, 3)      # an inaccurate answer cannot pass
        res_score = max(1, min(5, res_score))
 
        return {
            "res_score": res_score,
            "passed": res_score >= PASS_SCORE,
            "accuracy": scores["accuracy"],
            "relevance": scores["relevance"],
            "completeness": scores["completeness"],
            "usefulness": scores["usefulness"],
            "metric_average": round(average, 2),
            "metric_feedback": {m: getattr(verdict, f"{m}_reasoning") for m in WEIGHTS},
            "covered_elements": list(verdict.covered_elements),
            "missing_elements": list(verdict.missing_elements),
            "strengths": list(verdict.strengths),
            "weaknesses": list(verdict.weaknesses),
            "suggestions": list(verdict.suggestions),
            "reasoning": verdict.summary,
            "error": None,
        }
    except Exception as e:
        return {
            "res_score": None, "passed": False,
            "accuracy": None, "relevance": None, "completeness": None, "usefulness": None,
            "metric_average": None, "metric_feedback": {},
            "covered_elements": [], "missing_elements": [],
            "strengths": [], "weaknesses": [], "suggestions": [],
            "reasoning": "", "error": str(e),
        }

In [16]:
def evaluate_tool_usage(messages_list, expected_tools):
    """
    Evaluate if the right tools were used, using the agent's messages and the expected tools.
 
    TOOL COMPLETENESS (0-1): were all the necessary tools used?
        = expected tools that were called / expected tools.
    TOOL APPROPRIATENESS (0-1): were the right tools selected, and used sensibly?
        = share of the tools called that were expected, minus 0.1 for each repeated identical call
          (max 0.3) and 0.1 for each invalid argument (max 0.3).
    tool_score is the average of the two. passed means every expected tool was called and
    appropriateness is at least 0.5. If no tool calls can be read from the input at all, parse_warning
    says so (a zero score then reflects the input, not the agent). Failed or empty tool results are reported in the feedback
    but do not change the scores, because they are often the tool's fault, not the agent's.
    """
    import json
    import re
    from datetime import datetime
 
    PASS_APPROPRIATENESS = 0.5
    TOOL_PURPOSES = {
        "get_weather_forecast": "the forecast needed for timing, setpoint and solar advice",
        "get_electricity_prices": "the hourly prices needed to find cheap and expensive hours",
        "query_energy_usage": "the household's actual usage by device",
        "query_solar_generation": "the solar production data",
        "get_recent_energy_summary": "a summary of recent usage and generation",
        "search_energy_tips": "best-practice guidance from the tips knowledge base",
        "calculate_energy_savings": "a reliable savings calculation",
    }
 
    # Accept a result dict like {"messages": [...]}, a list of messages, a list of tool names, or a
    # list of tool-call dicts such as {"name": "get_weather_forecast", "args": {...}}
    if isinstance(messages_list, dict):
        messages_list = messages_list.get("messages", messages_list.get("tools_used", []))
    if isinstance(messages_list, str):
        messages_list = [t for t in re.split(r"[,;\n]", messages_list) if t.strip()]
    items = list(messages_list or [])
 
    # ---- 1. Read the tool calls and their results from the input ----
    calls, results, kinds = [], {}, set()
    for msg in items:
        if isinstance(msg, str):                                   # a tool name
            tool_name = msg.strip().split("(")[0].strip()
            if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_.\-]*", tool_name):     # looks like a tool name, not message text
                calls.append({"tool": tool_name, "args": None, "id": None})
                kinds.add("tool names")
            continue
 
        get = (lambda k: msg.get(k)) if isinstance(msg, dict) else (lambda k: getattr(msg, k, None))
        msg_type, tool_calls = get("type") or get("role"), get("tool_calls")
        name, content, call_id, status = get("name"), get("content"), get("tool_call_id"), get("status")
 
        is_message = bool(tool_calls) or content is not None or msg_type in (
            "ai", "assistant", "tool", "human", "user", "system")
        if not is_message:                                         # a bare tool call, not a message
            bare = name or get("tool") or get("tool_name")
            if bare:
                bare_args = get("args")
                calls.append({"tool": bare, "args": bare_args if isinstance(bare_args, dict) else None, "id": get("id")})
                kinds.add("tool-call dicts")
            continue
        kinds.add("messages")
 
        for call in tool_calls or []:
            if isinstance(call, dict) and "function" in call:          # OpenAI-style tool call
                fn = call["function"]
                call_name, call_args, cid = fn.get("name"), fn.get("arguments"), call.get("id")
                if isinstance(call_args, str):
                    try:
                        call_args = json.loads(call_args)
                    except ValueError:
                        call_args = {}
            elif isinstance(call, dict):
                call_name, call_args, cid = call.get("name"), call.get("args"), call.get("id")
            else:
                call_name, call_args, cid = getattr(call, "name", None), getattr(call, "args", None), getattr(call, "id", None)
            if call_name:
                calls.append({"tool": call_name, "args": call_args or {}, "id": cid})
 
        if msg_type == "tool":
            results[call_id] = {"name": name, "content": content, "status": status}
            if call_id is not None and not any(c["id"] == call_id for c in calls) and name:
                calls.append({"tool": name, "args": None, "id": call_id})   # tool message without a matching call
            elif call_id is None and name and not calls:
                calls.append({"tool": name, "args": None, "id": None})
 
    # Nothing in the input was recognized: report it instead of silently scoring zero
    input_format = ", ".join(sorted(kinds)) or ("empty" if not items else "unreadable")
    parse_warning = None
    if items and not kinds:
        parse_warning = (f"Could not read any tool calls from the input ({len(items)} item(s) of type "
                         f"{type(items[0]).__name__}). Pass the agent's message list, result['messages'], "
                         f"or a list of tool names.")
 
    # ---- 2. Check each call: did it return something, was it a repeat, were the arguments valid ----
    seen, redundant, failed, argument_issues = set(), 0, [], []
    for c in calls:
        c["repeat"] = False
        if c["args"] is not None:           # repeats can only be judged when the arguments are known
            key = (c["tool"], json.dumps(c["args"], sort_keys=True, default=str))
            c["repeat"] = key in seen
            redundant += c["repeat"]
            seen.add(key)
 
        res = results.get(c["id"])
        c["status"] = "no_result"
        if res is not None:
            text = res["content"] if isinstance(res["content"], str) else json.dumps(res["content"], default=str)
            stripped = (text or "").strip()
            reason = None
            if res.get("status") == "error":
                reason = stripped[:100] or "tool reported an error"
            elif stripped in ("", "null", "None", "{}", "[]"):
                reason = "empty result"
            elif stripped[:12].lower().startswith(('{"error"', "{'error'", "error")):
                reason = stripped[:100]
            c["status"] = "failed" if reason else "ok"
            if reason:
                failed.append(f"{c['tool']} ({reason})")
 
        args = c["args"]
        if isinstance(args, dict):          # arguments are unknown for bare tool messages, so skip the checks
            for k, v in args.items():
                if "date" in k and isinstance(v, str):
                    try:
                        datetime.strptime(v, "%Y-%m-%d")
                    except ValueError:
                        argument_issues.append(f"{c['tool']}: '{k}' is {v!r}, expected YYYY-MM-DD")
            if c["tool"] == "get_weather_forecast":
                if not str(args.get("location", "")).strip():
                    argument_issues.append("get_weather_forecast: no location was given")
                if "days" in args and not (isinstance(args["days"], int) and 1 <= args["days"] <= 7):
                    argument_issues.append(f"get_weather_forecast: days={args['days']!r} is outside 1-7")
            if isinstance(args.get("start_date"), str) and isinstance(args.get("end_date"), str):
                if args["start_date"] > args["end_date"]:
                    argument_issues.append(f"{c['tool']}: start_date is after end_date")
 
    # ---- 3. Compare with the expected tools ----
    used = list(dict.fromkeys(c["tool"] for c in calls))
    if isinstance(expected_tools, str):                       # e.g. "get_weather_forecast, get_electricity_prices"
        expected_tools = re.split(r"[,;\n]", expected_tools)
    expected = list(dict.fromkeys(t.strip() for t in (expected_tools or []) if isinstance(t, str) and t.strip()))
    hits = [t for t in expected if t in used]
    missing = [t for t in expected if t not in used]
    unexpected = [t for t in used if t not in expected]
 
    completeness = len(hits) / len(expected) if expected else 1.0
    if used:
        precision = len(hits) / len(used)
    else:
        precision = 1.0 if not expected else 0.0
    efficiency_penalty = min(0.3, 0.1 * redundant)
    argument_penalty = min(0.3, 0.1 * len(argument_issues))
    appropriateness = max(0.0, precision - efficiency_penalty - argument_penalty)
    tool_score = (completeness + appropriateness) / 2
    passed = not missing and appropriateness >= PASS_APPROPRIATENESS
 
    # ---- 4. Write the feedback ----
    strengths, weaknesses, suggestions = [], [], []
    if parse_warning:
        weaknesses.append(parse_warning)
        suggestions.append("Fix the evaluation input first: the tool scores are not meaningful until tool calls can be read.")
 
    if not expected:
        completeness_fb = "No tools were required for this question." + (" None were needed and none were called." if not used else "")
    elif not missing:
        completeness_fb = f"All {len(expected)} expected tool(s) were called: {', '.join(expected)}."
        strengths.append("Called every tool the task needs, so the answer can be based on real data.")
    else:
        lost = "; ".join(f"{t} ({TOOL_PURPOSES.get(t, 'its data')})" for t in missing)
        completeness_fb = (f"{len(hits)} of {len(expected)} expected tools were called. "
                           f"Missing: {lost}. Without them the answer cannot rely on that data.")
        weaknesses.append(f"Did not call: {', '.join(missing)}.")
        for t in missing:
            suggestions.append(f"Make the system prompt require {t} for this kind of question "
                               f"({TOOL_PURPOSES.get(t, 'to get the data it provides')}).")
 
    if not used and expected:
        appropriateness_fb = "No tools were called, so nothing could be selected or checked."
        weaknesses.append("The agent answered without using any tool.")
        suggestions.append("Tell the agent never to answer from memory when tools can supply the data.")
    elif not used:
        appropriateness_fb = "No tools were used and none were expected."
    else:
        parts = [f"{len(hits)} of {len(used)} tool(s) called were expected (selection {precision:.2f})."]
        if unexpected:
            parts.append(f"Extra tool(s) called: {', '.join(unexpected)}. This is fine if they added relevant data; "
                         f"if so, add them to expected_tools.")
            weaknesses.append(f"Called tools that were not expected: {', '.join(unexpected)}.")
            suggestions.append("If the extra tools are not needed, narrow the tool-use rules in the system prompt.")
        else:
            parts.append("Every tool called was an expected one.")
            strengths.append("Selected only the tools the task needed.")
        if redundant:
            parts.append(f"{redundant} repeated identical call(s) (-{efficiency_penalty:.1f}).")
            weaknesses.append(f"{redundant} tool call(s) were exact repeats of an earlier call.")
            suggestions.append("Tell the agent to reuse results it already has instead of calling a tool again.")
        else:
            strengths.append("No repeated tool calls.")
        if argument_issues:
            parts.append(f"{len(argument_issues)} invalid argument(s) (-{argument_penalty:.1f}): " + "; ".join(argument_issues) + ".")
            weaknesses.append("Invalid tool arguments: " + "; ".join(argument_issues) + ".")
            suggestions.append("Show the argument formats in the prompt, for example dates as YYYY-MM-DD and forecast days from 1 to 7.")
        appropriateness_fb = " ".join(parts)
 
    if failed:
        reliability_fb = (f"{len(failed)} tool call(s) failed or returned nothing: {'; '.join(failed)}. "
                          f"This is not scored, but the answer may rest on missing data.")
        weaknesses.append("Tool calls that failed or returned nothing: " + "; ".join(failed) + ".")
        suggestions.append("Check that those tools are implemented and return data. Fix the tool before changing the prompt.")
    elif calls:
        reliability_fb = "Every tool call returned a usable result."
        strengths.append("All tool calls returned usable results.")
    else:
        reliability_fb = "No tool calls to check."
 
    band = "excellent" if tool_score >= 0.9 else "good" if tool_score >= 0.7 else "fair" if tool_score >= 0.5 else "poor"
    summary = ((parse_warning + " ") if parse_warning else "") + (f"Tool usage is {band} (score {tool_score:.2f}): completeness {completeness:.2f}, "
               f"appropriateness {appropriateness:.2f}. "
               + ("Called in this order: " + " -> ".join(c["tool"] for c in calls) + "." if calls else "No tools were called."))
 
    return {
        "tools_used": used,
        "expected_tools": expected,
        "missing_tools": missing,
        "unexpected_tools": unexpected,
        "recall": round(completeness, 3),
        "precision": round(precision, 3),
        "tool_completeness": round(completeness, 3),
        "tool_appropriateness": round(appropriateness, 3),
        "tool_score": round(tool_score, 3),
        "passed": passed,
        "tool_calls": [{"tool": c["tool"], "args": c["args"], "status": c["status"], "repeat": c["repeat"]} for c in calls],
        "redundant_calls": redundant,
        "failed_calls": failed,
        "argument_issues": argument_issues,
        "tool_feedback": {
            "appropriateness": appropriateness_fb,
            "completeness": completeness_fb,
            "reliability": reliability_fb,
            "input": parse_warning or f"Read {len(calls)} tool call(s) from a {input_format} input.",
        },
        "tool_strengths": strengths,
        "tool_weaknesses": weaknesses,
        "tool_suggestions": suggestions,
        "tool_summary": summary,
        "input_format": input_format,
        "parse_warning": parse_warning,
    }

In [17]:
# TODO: Generate a comprehensive evaluation report
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement
def generate_evaluation_report(results):
    """
    Build a structured evaluation report (a dict) from the nested results. Nothing is printed here:
    pass the report to display_evaluation_report() to show it.
 
    Scores:
      combined score per case = 0.6 x response score (1-5 rescaled to 0-1) + 0.4 x tool score (0-1)
      overall score           = average combined score x 100, graded A (90+) to F (below 60)
    A case passes when both its response and its tool usage pass. Cases where the judge model failed
    are listed as errors and left out of the scores and pass rates, so a judge failure is never
    mistaken for a weak answer.
    """
    from collections import Counter, defaultdict
    from datetime import datetime
 
    RESPONSE_WEIGHT, TOOL_WEIGHT = 0.6, 0.4
    RESPONSE_PASS = 4
    METRICS = ["accuracy", "relevance", "completeness", "usefulness"]
    TOOL_HINTS = {
        "get_weather_forecast": "check the forecast before giving timing, setpoint or solar advice",
        "get_electricity_prices": "fetch hourly prices before recommending when to run any load",
        "query_energy_usage": "look up the household's actual usage by device before estimating savings",
        "query_solar_generation": "check solar generation data when advising on solar use",
        "get_recent_energy_summary": "start from the recent usage summary for questions about current usage",
        "search_energy_tips": "search the tips knowledge base and base advice on what it finds",
        "calculate_energy_savings": "use the savings calculator for every savings figure",
    }
    METRIC_ADVICE = {
        "accuracy": "Require every number to come from a tool result, run all savings math through "
                    "calculate_energy_savings, and say when a figure is an estimate.",
        "relevance": "Tell the agent to answer the exact question first (right device, time frame and goal) "
                     "and only then add supporting detail.",
        "completeness": "Add a checklist to the prompt so every answer covers timing, cost analysis, "
                        "solar or weather considerations and estimated savings when they apply.",
        "usefulness": "Require specific devices and time windows, estimated kWh and dollar savings, "
                      "and a priority order in every recommendation.",
    }
 
    if not results:
        return {"error": "No results to report."}
 
    # ------------------------------------------------------------------
    # 1. One row per test case
    # ------------------------------------------------------------------
    rows = []
    for i, r in enumerate(results):
        resp, tool = r.get("response_eval") or {}, r.get("tool_eval") or {}
        case_id = str(r.get("id") or f"case_{i + 1}")
        head, _, tail = case_id.rpartition("_")
        category = head if head and tail.isdigit() else case_id
 
        res_score = resp.get("res_score", resp.get("score"))
        judged = res_score is not None and not resp.get("error")
        resp_norm = (res_score - 1) / 4 if judged else 0.0
        missing_tools = tool.get("missing_tools") or []
        t_complete = tool.get("tool_completeness", tool.get("recall", 0.0))
        t_appropriate = tool.get("tool_appropriateness", tool.get("precision", 0.0))
        t_score = tool.get("tool_score", tool.get("score", t_complete))
        resp_passed = bool(judged and resp.get("passed", res_score >= RESPONSE_PASS))
        tool_passed = bool(tool.get("passed", not missing_tools))
 
        if not judged:
            status = "ERROR (judge)"
        elif resp_passed and tool_passed:
            status = "PASS"
        elif not resp_passed and not tool_passed:
            status = "FAIL (response + tools)"
        elif not resp_passed:
            status = "FAIL (response)"
        else:
            status = "FAIL (tools)"
 
        rows.append({
            "id": case_id, "category": category, "status": status, "judged": judged,
            "response_score": res_score if judged else None,
            **{m: (resp.get(m) if judged else None) for m in METRICS},
            "tool_completeness": t_complete, "tool_appropriateness": t_appropriate, "tool_score": t_score,
            "combined": round(RESPONSE_WEIGHT * resp_norm + TOOL_WEIGHT * t_score, 3) if judged else None,
            "response_passed": resp_passed, "tool_passed": tool_passed,
            "passed": resp_passed and tool_passed,
            "judge_error": resp.get("error"),
            "response_summary": resp.get("reasoning", ""),
            "missing_elements": resp.get("missing_elements") or [],
            "response_weaknesses": resp.get("weaknesses") or [],
            "response_suggestions": resp.get("suggestions") or [],
            "tools_used": tool.get("tools_used") or [],
            "expected_tools": tool.get("expected_tools") or [],
            "missing_tools": missing_tools,
            "unexpected_tools": tool.get("unexpected_tools") or [],
            "failed_calls": tool.get("failed_calls") or [],
            "argument_issues": tool.get("argument_issues") or [],
            "redundant_calls": tool.get("redundant_calls", 0) or 0,
            "tool_summary": tool.get("tool_summary", ""),
        })
 
    n = len(rows)
    judged_rows = [x for x in rows if x["judged"]]
 
    # ------------------------------------------------------------------
    # 2. Overall scores
    # ------------------------------------------------------------------
    n_judged = len(judged_rows)
    avg_combined = sum(x["combined"] for x in judged_rows) / n_judged if n_judged else None
    overall_score = round(avg_combined * 100, 1) if avg_combined is not None else None
    if overall_score is None:
        grade, verdict = "N/A", "Not scored (the judge model failed on every case)"
    elif overall_score >= 90:
        grade, verdict = "A", "Excellent"
    elif overall_score >= 80:
        grade, verdict = "B", "Good"
    elif overall_score >= 70:
        grade, verdict = "C", "Satisfactory"
    elif overall_score >= 60:
        grade, verdict = "D", "Needs improvement"
    else:
        grade, verdict = "F", "Poor"
    avg_response = sum(x["response_score"] for x in judged_rows) / n_judged if n_judged else None
    overall = {
        "overall_score_out_of_100": overall_score,
        "grade": grade,
        "verdict": verdict,
        "pass_rate": round(sum(x["passed"] for x in judged_rows) / n_judged, 3) if n_judged else None,
        "response_pass_rate": round(sum(x["response_passed"] for x in judged_rows) / n_judged, 3) if n_judged else None,
        "tool_pass_rate": round(sum(x["tool_passed"] for x in rows) / n, 3),
        "avg_response_score_1_to_5": round(avg_response, 2) if avg_response is not None else None,
        "avg_tool_score_0_to_1": round(sum(x["tool_score"] for x in rows) / n, 3),
        "avg_combined_score_0_to_1": round(avg_combined, 3) if avg_combined is not None else None,
        "cases_judged": n_judged,
        "cases_passed": sum(x["passed"] for x in judged_rows),
        "cases_failed": sum(1 for x in rows if x["status"].startswith("FAIL")),
        "cases_with_judge_errors": n - n_judged,
    }
 
    # ------------------------------------------------------------------
    # 3. Response and tool metrics
    # ------------------------------------------------------------------
    response_metrics = {}
    for m in METRICS:
        vals = [x[m] for x in judged_rows if x[m] is not None]
        if vals:
            response_metrics[m] = {
                "average": round(sum(vals) / len(vals), 2), "min": min(vals), "max": max(vals),
                "share_scoring_4_or_more": round(sum(v >= 4 for v in vals) / len(vals), 2),
                "cases_scoring_2_or_less": [x["id"] for x in judged_rows if x[m] is not None and x[m] <= 2],
            }
    tool_metrics = {
        "tool_completeness": round(sum(x["tool_completeness"] for x in rows) / n, 3),
        "tool_appropriateness": round(sum(x["tool_appropriateness"] for x in rows) / n, 3),
        "tool_score": round(sum(x["tool_score"] for x in rows) / n, 3),
    }
 
    # ------------------------------------------------------------------
    # 4. Tool analysis
    # ------------------------------------------------------------------
    all_tools = sorted({t for x in rows for t in x["expected_tools"] + x["tools_used"]})
    tool_table = {
        t: {
            "expected_in_cases": sum(t in x["expected_tools"] for x in rows),
            "used_in_cases": sum(t in x["tools_used"] for x in rows),
            "missed_in_cases": sum(t in x["missing_tools"] for x in rows),
            "extra_in_cases": sum(t in x["unexpected_tools"] for x in rows),
        }
        for t in all_tools
    }
    failed_by_tool = Counter(f.split(" (")[0] for x in rows for f in x["failed_calls"])
    missed = Counter(t for x in rows for t in x["missing_tools"])
    extra = Counter(t for x in rows for t in x["unexpected_tools"])
    argument_issues = [f"{x['id']}: {a}" for x in rows for a in x["argument_issues"]]
    tool_analysis = {
        "by_tool": tool_table,
        "failed_calls_by_tool": dict(failed_by_tool),
        "failed_calls": [f"{x['id']}: {f}" for x in rows for f in x["failed_calls"]],
        "argument_issues": argument_issues,
        "redundant_calls": sum(x["redundant_calls"] for x in rows),
    }
    gaps = Counter(g.strip().lower() for x in rows for g in x["missing_elements"] if g and g.strip())
    common_gaps = [{"element": g, "cases": c} for g, c in gaps.most_common(5)]
 
    # ------------------------------------------------------------------
    # 5. Categories
    # ------------------------------------------------------------------
    groups = defaultdict(list)
    for x in rows:
        groups[x["category"]].append(x)
    category_breakdown = {}
    for cat, xs in groups.items():
        js = [x for x in xs if x["judged"]]
        category_breakdown[cat] = {
            "cases": len(xs),
            "judged": len(js),
            "pass_rate": round(sum(x["passed"] for x in js) / len(js), 2) if js else None,
            "avg_response_score": round(sum(x["response_score"] for x in js) / len(js), 2) if js else None,
            "avg_tool_score": round(sum(x["tool_score"] for x in xs) / len(xs), 2),
            "avg_combined": round(sum(x["combined"] for x in js) / len(js), 2) if js else None,
        }
 
    # ------------------------------------------------------------------
    # 6. Strengths and weaknesses
    # ------------------------------------------------------------------
    strengths, weaknesses = [], []
    for m, s in response_metrics.items():
        if s["average"] >= 4.2:
            strengths.append(f"{m.capitalize()} is strong (average {s['average']}/5, "
                             f"{s['share_scoring_4_or_more']:.0%} of cases score 4 or more).")
        elif s["average"] < 3.5:
            weaknesses.append(f"{m.capitalize()} is weak (average {s['average']}/5; "
                              f"{len(s['cases_scoring_2_or_less'])} case(s) score 2 or lower).")
        elif s["average"] < 4.0:
            weaknesses.append(f"{m.capitalize()} is only fair (average {s['average']}/5).")
    if tool_metrics["tool_completeness"] >= 0.9:
        strengths.append(f"The agent calls the tools each task needs "
                         f"(tool completeness {tool_metrics['tool_completeness']:.2f}).")
    elif tool_metrics["tool_completeness"] < 0.7:
        weaknesses.append(f"The agent often skips needed tools (tool completeness "
                          f"{tool_metrics['tool_completeness']:.2f}).")
    if tool_metrics["tool_appropriateness"] >= 0.9:
        strengths.append(f"Tool selection is precise "
                         f"(tool appropriateness {tool_metrics['tool_appropriateness']:.2f}).")
    elif tool_metrics["tool_appropriateness"] < 0.7:
        weaknesses.append(f"Tool selection is noisy: extra, repeated or badly specified calls "
                          f"(tool appropriateness {tool_metrics['tool_appropriateness']:.2f}).")
    if failed_by_tool:
        weaknesses.append("Tool calls returned errors or nothing: "
                          + ", ".join(f"{t} x{c}" for t, c in failed_by_tool.most_common()) + ".")
    ranked_cats = sorted([kv for kv in category_breakdown.items() if kv[1]["avg_combined"] is not None],
                         key=lambda kv: kv[1]["avg_combined"], reverse=True)
    if len(ranked_cats) > 1:
        strengths.append(f"Best category: '{ranked_cats[0][0]}' (combined {ranked_cats[0][1]['avg_combined']:.2f}).")
        if ranked_cats[-1][1]["avg_combined"] < 0.7:
            weaknesses.append(f"Weakest category: '{ranked_cats[-1][0]}' "
                              f"(combined {ranked_cats[-1][1]['avg_combined']:.2f}).")
    if overall["cases_with_judge_errors"]:
        weaknesses.append(f"{overall['cases_with_judge_errors']} case(s) could not be judged because the judge model failed.")
 
    # ------------------------------------------------------------------
    # 7. Recommendations
    # ------------------------------------------------------------------
    recs = []
    if overall["cases_with_judge_errors"]:
        recs.append({"priority": "high", "area": "Evaluation setup",
                     "action": "Check the judge model name and API key, then re-run the evaluation.",
                     "evidence": "Judge errors in: " + ", ".join(x["id"] for x in rows if not x["judged"]) + "."})
    for t, c in failed_by_tool.most_common():
        recs.append({"priority": "high", "area": "Tool implementation",
                     "action": f"Fix '{t}': it must return real data before any prompt change will help.",
                     "evidence": f"{c} call(s) failed or returned nothing."})
    for t, c in missed.most_common():
        recs.append({"priority": "high" if c / n >= 0.3 else "medium", "area": "System prompt: tool use",
                     "action": f"Instruct the agent to {TOOL_HINTS.get(t, 'use this tool when relevant')}.",
                     "evidence": f"'{t}' was expected but not called in {c} of {n} cases."})
    for m, s in response_metrics.items():
        if s["average"] < 4.0:
            gap_text = ""
            if m == "completeness" and common_gaps:
                gap_text = " Most common gaps: " + "; ".join(f"{g['element']} ({g['cases']})" for g in common_gaps) + "."
            recs.append({"priority": "high" if s["average"] < 3.5 else "medium",
                         "area": f"System prompt: {m}", "action": METRIC_ADVICE[m],
                         "evidence": f"Average {m} is {s['average']}/5.{gap_text}"})
    if argument_issues:
        recs.append({"priority": "medium", "area": "System prompt: tool arguments",
                     "action": "Show the argument formats in the prompt, for example dates as YYYY-MM-DD "
                               "and forecast days from 1 to 7.",
                     "evidence": f"{len(argument_issues)} invalid argument(s), e.g. {argument_issues[0]}"})
    if extra:
        recs.append({"priority": "low", "area": "Test design or prompt",
                     "action": "If the extra tools add useful data, add them to expected_tools. "
                               "If not, narrow the tool-use rules in the prompt.",
                     "evidence": "Unexpected tools called: " + ", ".join(f"{t} x{c}" for t, c in extra.most_common()) + "."})
    if tool_analysis["redundant_calls"]:
        recs.append({"priority": "low", "area": "System prompt: efficiency",
                     "action": "Tell the agent to reuse tool results it already has instead of calling again.",
                     "evidence": f"{tool_analysis['redundant_calls']} repeated identical call(s)."})
    weak_cats = [c for c, s in category_breakdown.items() if s["avg_combined"] is not None and s["avg_combined"] < 0.7]
    if weak_cats:
        recs.append({"priority": "medium", "area": "Weak scenarios",
                     "action": "Read the transcripts of these categories and add targeted instructions or examples: "
                               + ", ".join(weak_cats) + ".",
                     "evidence": "Combined score below 0.70."})
    if not recs:
        recs.append({"priority": "low", "area": "Test coverage",
                     "action": "All checks passed. Add harder cases (missing data, tool failures, off-topic questions).",
                     "evidence": "No weaknesses found in this test set."})
    recs.sort(key=lambda x: {"high": 0, "medium": 1, "low": 2}[x["priority"]])
 
    # ------------------------------------------------------------------
    # 8. Cases needing attention, executive summary, final structure
    # ------------------------------------------------------------------
    attention = [
        {k: x[k] for k in ("id", "status", "combined", "response_summary", "missing_elements", "response_weaknesses",
                           "response_suggestions", "missing_tools", "unexpected_tools", "failed_calls",
                           "argument_issues", "tool_summary")}
        for x in sorted(rows, key=lambda r: -1 if r["combined"] is None else r["combined"]) if x["status"] != "PASS"
    ]
    dims = {f"response {m}": s["average"] / 5 for m, s in response_metrics.items()}
    dims["tool completeness"] = tool_metrics["tool_completeness"]
    dims["tool appropriateness"] = tool_metrics["tool_appropriateness"]
    best, worst = max(dims, key=dims.get), min(dims, key=dims.get)
    resp_text = f"{avg_response:.2f}/5" if avg_response is not None else "n/a"
    high = len([r for r in recs if r["priority"] == "high"])
    if n_judged:
        summary = (f"The agent passed {overall['cases_passed']} of {n_judged} judged test cases ({overall['pass_rate']:.0%}) "
                   f"with an overall score of {overall_score}/100 (grade {grade}, {verdict.lower()}). "
                   f"Response quality averaged {resp_text} and tool usage {tool_metrics['tool_score']:.2f}/1. "
                   f"The strongest area is {best} and the weakest is {worst}. ")
    else:
        summary = ("No test case could be judged, so no overall score is available. "
                   f"Tool usage averaged {tool_metrics['tool_score']:.2f}/1. ")
    if overall["cases_with_judge_errors"]:
        summary += f"{overall['cases_with_judge_errors']} case(s) could not be judged and are excluded from the scores. "
    summary += f"{high} high-priority recommendation(s) follow."
 
    return {
        "metadata": {"generated_at": datetime.now().strftime("%Y-%m-%d %H:%M"), "total_cases": n,
                     "weights": {"response": RESPONSE_WEIGHT, "tools": TOOL_WEIGHT},
                     "response_pass_score": RESPONSE_PASS},
        "summary": summary,
        "overall": overall,
        "response_metrics": response_metrics,
        "tool_metrics": tool_metrics,
        "tool_analysis": tool_analysis,
        "common_gaps": common_gaps,
        "category_breakdown": category_breakdown,
        "test_cases": rows,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "recommendations": recs,
        "attention_cases": attention,
    }

In [18]:

def display_evaluation_report(report):
    """Print the report from generate_evaluation_report() in a readable, sectioned layout."""
    import textwrap
 
    if not report or "error" in report:
        print(report.get("error", "No report to display.") if report else "No report to display.")
        return
 
    W = 96
    meta, ov = report["metadata"], report["overall"]
 
    print("=" * W)
    print("ECOHOME ENERGY ADVISOR: EVALUATION REPORT")
    print(f"Generated {meta['generated_at']}  |  {meta['total_cases']} test cases  |  "
          f"weights: response {meta['weights']['response']:.0%}, tools {meta['weights']['tools']:.0%}")
    print("=" * W)
 
    print("\n1. EXECUTIVE SUMMARY")
    print("-" * W)
    print(textwrap.fill(report["summary"], W, initial_indent="  ", subsequent_indent="  "))
 
    print("\n2. OVERALL SCORES")
    print("-" * W)
    resp_avg = f"{ov['avg_response_score_1_to_5']:.2f} / 5" if ov["avg_response_score_1_to_5"] is not None else "n/a"
    score_text = f"{ov['overall_score_out_of_100']:.1f} / 100" if ov["overall_score_out_of_100"] is not None else "n/a"
    pass_text = f"{ov['pass_rate']:.0%}" if ov["pass_rate"] is not None else "n/a"
    resp_pass_text = f"{ov['response_pass_rate']:.0%}" if ov["response_pass_rate"] is not None else "n/a"
    print(f"  Overall score            {score_text}   grade {ov['grade']} ({ov['verdict']})")
    print(f"  Cases passed             {ov['cases_passed']} / {ov['cases_judged']} judged     pass rate {pass_text}")
    print(f"  Response pass rate       {resp_pass_text}")
    print(f"  Tool usage pass rate     {ov['tool_pass_rate']:>6.0%}")
    print(f"  Average response score   {resp_avg}")
    print(f"  Average tool score       {ov['avg_tool_score_0_to_1']:.2f} / 1")
    if ov["cases_with_judge_errors"]:
        print(f"  Judge errors             {ov['cases_with_judge_errors']} case(s) not judged, excluded from scores and pass rates")
 
    print("\n3. RESPONSE QUALITY (scores 1-5)")
    print("-" * W)
    print(f"  {'metric':<15}{'average':>8}{'min':>6}{'max':>6}{'>=4':>7}   scale")
    for m, s in report["response_metrics"].items():
        fill = round(s["average"] / 5 * 20)
        print(f"  {m.capitalize():<15}{s['average']:>8.2f}{s['min']:>6}{s['max']:>6}"
              f"{s['share_scoring_4_or_more']:>7.0%}   [{'#' * fill}{'.' * (20 - fill)}]")
 
    print("\n4. TOOL USAGE (scores 0-1)")
    print("-" * W)
    for label, key in (("Completeness", "tool_completeness"), ("Appropriateness", "tool_appropriateness"),
                       ("Tool score", "tool_score")):
        v = report["tool_metrics"][key]
        fill = round(v * 20)
        print(f"  {label:<17}{v:>6.2f}   [{'#' * fill}{'.' * (20 - fill)}]")
    print(f"\n  {'tool':<28}{'expected':>9}{'used':>6}{'missed':>8}{'extra':>7}")
    for t, c in report["tool_analysis"]["by_tool"].items():
        print(f"  {t:<28}{c['expected_in_cases']:>9}{c['used_in_cases']:>6}{c['missed_in_cases']:>8}{c['extra_in_cases']:>7}")
    ta = report["tool_analysis"]
    if ta["failed_calls"]:
        print("\n  Failed or empty tool calls:")
        for f in ta["failed_calls"]:
            print(textwrap.fill(f, W, initial_indent="    - ", subsequent_indent="      "))
    if ta["argument_issues"]:
        print("\n  Invalid arguments:")
        for a in ta["argument_issues"]:
            print(textwrap.fill(a, W, initial_indent="    - ", subsequent_indent="      "))
    if ta["redundant_calls"]:
        print(f"\n  Repeated identical calls: {ta['redundant_calls']}")
 
    print("\n5. PERFORMANCE BY CATEGORY")
    print("-" * W)
    print(f"  {'category':<26}{'cases':>6}{'pass':>7}{'response':>10}{'tools':>8}{'combined':>10}")
    for cat, s in report["category_breakdown"].items():
        rs = f"{s['avg_response_score']:.2f}" if s["avg_response_score"] is not None else "n/a"
        pr = f"{s['pass_rate']:.0%}" if s["pass_rate"] is not None else "n/a"
        cb = f"{s['avg_combined']:.2f}" if s["avg_combined"] is not None else "n/a"
        print(f"  {cat:<26}{s['cases']:>6}{pr:>7}{rs:>10}{s['avg_tool_score']:>8.2f}{cb:>10}")
 
    print("\n6. TEST CASE RESULTS")
    print("-" * W)
    print(f"  {'id':<24}{'resp':>5}{'acc':>5}{'rel':>5}{'comp':>5}{'use':>5}{'tools':>7}{'comb.':>7}  status")
    for x in report["test_cases"]:
        cells = ["-" if x[k] is None else str(x[k]) for k in ("response_score", "accuracy", "relevance", "completeness", "usefulness")]
        print(f"  {x['id']:<24}{cells[0]:>5}{cells[1]:>5}{cells[2]:>5}{cells[3]:>5}{cells[4]:>5}"
              f"{x['tool_score']:>7.2f}{('-' if x['combined'] is None else format(x['combined'], '.2f')):>7}  {x['status']}")
 
    print("\n7. STRENGTHS")
    print("-" * W)
    for s in report["strengths"] or ["None identified yet."]:
        print(textwrap.fill(s, W, initial_indent="  + ", subsequent_indent="    "))
 
    print("\n8. WEAKNESSES")
    print("-" * W)
    for w in report["weaknesses"] or ["None identified."]:
        print(textwrap.fill(w, W, initial_indent="  - ", subsequent_indent="    "))
    if report["common_gaps"]:
        print("\n  Most common missing elements in answers:")
        for g in report["common_gaps"]:
            print(f"    - {g['element']} ({g['cases']} case(s))")
 
    print("\n9. RECOMMENDATIONS (highest priority first)")
    print("-" * W)
    for i, r in enumerate(report["recommendations"], 1):
        print(textwrap.fill(f"{i}. [{r['priority'].upper()}] {r['area']}: {r['action']}", W,
                            initial_indent="  ", subsequent_indent="     "))
        print(textwrap.fill(f"Evidence: {r['evidence']}", W, initial_indent="     ", subsequent_indent="     "))
 
    if report["attention_cases"]:
        print("\n10. CASES NEEDING ATTENTION (lowest score first)")
        print("-" * W)
        for c in report["attention_cases"]:
            combined_text = "not scored" if c["combined"] is None else f"combined {c['combined']:.2f}"
            print(f"  {c['id']}  [{c['status']}]  {combined_text}")
            if c["response_summary"]:
                print(textwrap.fill("Response: " + c["response_summary"], W, initial_indent="    ", subsequent_indent="    "))
            for label, key in (("Missing in answer", "missing_elements"), ("Problems", "response_weaknesses"),
                               ("Missing tools", "missing_tools"), ("Extra tools", "unexpected_tools"),
                               ("Failed calls", "failed_calls"), ("Argument issues", "argument_issues"),
                               ("Suggested fixes", "response_suggestions")):
                if c[key]:
                    print(textwrap.fill(f"{label}: " + "; ".join(c[key]), W, initial_indent="    ", subsequent_indent="      "))
            print()
 
    print("=" * W)

In [19]:
response = []
for i in range(len(test_results)):
    res = test_results[i]
    res_eval = evaluate_response(res['question'], res['response'], res['expected_response'])
    tool_eval = evaluate_tool_usage(res['tools'], res['expected_tools'])
 
    response.append(
        {
            'id': res['test_id'],
            'response_eval': res_eval,
            'tool_eval': tool_eval
        }
    )
    print(f"{res['test_id']:<26} response {res_eval['res_score']}/5 | tools {tool_eval['tool_score']:.2f}")

ev_charging_1              response 4/5 | tools 0.83
ev_charging_2              response 5/5 | tools 0.50
thermostat_1               response 4/5 | tools 0.50
thermostat_2               response 4/5 | tools 0.83
appliance_scheduling_1     response 4/5 | tools 0.42
appliance_scheduling_2     response 5/5 | tools 0.75
solar_maximization_1       response 4/5 | tools 0.75
solar_maximization_2       response 4/5 | tools 0.83
cost_savings_1             response 4/5 | tools 0.50
cost_savings_2             response 3/5 | tools 1.00


In [20]:
report = generate_evaluation_report(response)
display_evaluation_report(report)

ECOHOME ENERGY ADVISOR: EVALUATION REPORT
Generated 2026-10-29 15:41  |  10 test cases  |  weights: response 60%, tools 40%

1. EXECUTIVE SUMMARY
------------------------------------------------------------------------------------------------
  The agent passed 3 of 10 judged test cases (30%) with an overall score of 74.2/100 (grade C,
  satisfactory). Response quality averaged 4.10/5 and tool usage 0.69/1. The strongest area is
  response relevance and the weakest is tool appropriateness. 1 high-priority recommendation(s)
  follow.

2. OVERALL SCORES
------------------------------------------------------------------------------------------------
  Overall score            74.2 / 100   grade C (Satisfactory)
  Cases passed             3 / 10 judged     pass rate 30%
  Response pass rate       90%
  Tool usage pass rate        40%
  Average response score   4.10 / 5
  Average tool score       0.69 / 1

3. RESPONSE QUALITY (scores 1-5)
----------------------------------------------------

In [21]:
import json
 
with open("evaluation_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)
print("Saved evaluation_report.json")

Saved evaluation_report.json
